In [ ]:
# ─── Cell: ACF/PACF Analysis & Curve Segment Regime Filter ───────────────────
# Purpose: determine natural mean reversion horizon per tenor and whether
# the 20-day PCA window and 60-day Z-score window are appropriate.
# ──────────────────────────────────────────────────────────────────────────────

# ──────────────────────────────────────────────────────────────────────────────
# ANOTHER TEST FILE TO EVALUAE THE AVG MEAN REVERSION FOR EACH TENORS
# RESIDUALS_DF COMES FROM THE MAIN PRINCIPAL COMPONENT ANALYSIS,IPYNB FILE
# ──────────────────────────────────────────────────────────────────────────────


MAX_LAGS = 30
tenors   = ['1Mo', '3Mo', '6Mo', '1Yr', '2Yr', '3Yr', '5Yr', '7Yr', '10Yr', '30Yr']

# ── 1. Compute ACF + PACF ─────────────────────────────────────────────────────
acf_results  = {}
pacf_results = {}
ci_bounds    = {}

for tenor in tenors:
    series = residuals_df[tenor].dropna()
    acf_results[tenor]  = acf(series,  nlags=MAX_LAGS, fft=True)
    pacf_results[tenor] = pacf(series, nlags=MAX_LAGS, method='ywa')
    ci_bounds[tenor]    = 1.96 / np.sqrt(len(series))


# ── 2. Summary table helpers ──────────────────────────────────────────────────
def _first_acf_crossing(acf_vals, ci):
    """First lag ≥ 1 where |ACF| drops below the 95% CI bound."""
    return next(
        (lag for lag in range(1, len(acf_vals)) if abs(acf_vals[lag]) < ci),
        None
    )

def _decay_type(acf_vals, first_crossing, ci):
    """
    Sharp Cutoff — ACF crosses CI within lag 4 AND the drop is abrupt
                   (previous lag was still significant).
    Gradual Decay — ACF tapers slowly, consistent with AR persistence.
    Never Crosses  — possible unit-root or very long memory behaviour.
    """
    if first_crossing is None:
        return 'Never Crosses CI'
    if first_crossing <= 4 and (first_crossing == 1 or abs(acf_vals[first_crossing - 1]) > ci):
        return 'Sharp Cutoff'
    return 'Gradual Decay'

def _classify_acf(first_crossing):
    if first_crossing is None:
        return 'Unit Root — Do Not Trade'
    if first_crossing <= 5:
        return 'Fast (lags 1-5) — High Signal Quality'
    if first_crossing <= 10:
        return 'Moderate (lags 6-10) — Tradeable'
    return 'Slow (lag >10) — Caution'


# ── 3. Build and print summary table ─────────────────────────────────────────
summary_rows = []
for tenor in tenors:
    acf_vals  = acf_results[tenor]
    pacf_vals = pacf_results[tenor]
    ci = ci_bounds[tenor]

    first_crossing = _first_acf_crossing(acf_vals, ci)
    sig_pacf_lags  = [lag for lag in range(1, MAX_LAGS + 1) if abs(pacf_vals[lag]) > ci]

    summary_rows.append({
        'Tenor':           tenor,
        'First ACF Cross': first_crossing if first_crossing is not None else 'Never',
        'Decay Type':      _decay_type(acf_vals, first_crossing, ci),
        'Sig. PACF Lags':  sig_pacf_lags if sig_pacf_lags else ['None'],
        'Classification':  _classify_acf(first_crossing),
    })

acf_summary_df = pd.DataFrame(summary_rows).set_index('Tenor')

print("=" * 90)
print("ACF/PACF Analysis — Mean Reversion Horizon per Tenor (Daily PCA Residuals)")
print("95% CI: ±1.96/√n  |  Max lags: 30  |  ACF: FFT  |  PACF: Yule-Walker (unbiased)")
print("=" * 90)
print(acf_summary_df.to_string())
print("=" * 90)


# ── 4. ACF/PACF subplots — 10 rows × 2 cols (ACF left | PACF right) ──────────
lags = list(range(1, MAX_LAGS + 1))

subplot_titles = []
for tenor in tenors:
    subplot_titles += [f'<b>{tenor}</b> — ACF', f'<b>{tenor}</b> — PACF']

fig_acf_pacf = make_subplots(
    rows=10, cols=2,
    subplot_titles=subplot_titles,
    shared_xaxes=False,
    vertical_spacing=0.025,
    horizontal_spacing=0.10
)

for row, tenor in enumerate(tenors, start=1):
    acf_plot  = acf_results[tenor][1:]    # lags 1..30 (drop lag-0 = 1.0)
    pacf_plot = pacf_results[tenor][1:]   # lags 1..30
    ci = ci_bounds[tenor]

    # ACF bars (blue)
    fig_acf_pacf.add_trace(
        go.Bar(x=lags, y=acf_plot, marker_color='#1f77b4',
               showlegend=False,
               hovertemplate='Lag %{x}: %{y:.4f}<extra></extra>'),
        row=row, col=1
    )
    # PACF bars (orange)
    fig_acf_pacf.add_trace(
        go.Bar(x=lags, y=pacf_plot, marker_color='#ff7f0e',
               showlegend=False,
               hovertemplate='Lag %{x}: %{y:.4f}<extra></extra>'),
        row=row, col=2
    )
    # 95% CI bands — upper and lower — for both columns
    for col_idx in [1, 2]:
        for y_val in [ci, -ci]:
            fig_acf_pacf.add_trace(
                go.Scatter(
                    x=[0.5, MAX_LAGS + 0.5], y=[y_val, y_val],
                    mode='lines',
                    line=dict(color='#d62728', dash='dash', width=1),
                    showlegend=False, hoverinfo='skip'
                ),
                row=row, col=col_idx
            )

fig_acf_pacf.update_layout(
    title=dict(
        text=(
            '<b>ACF & PACF — Daily PCA Residuals by Tenor</b><br>'
            'Blue: ACF  |  Orange: PACF  |  Dashed red: 95% CI (±1.96/√n)'
        ),
        x=0.5, font=dict(size=18)
    ),
    template='plotly_white',
    height=2800,
    showlegend=False,
    margin=dict(l=50, r=50, t=120, b=50),
    bargap=0.15
)

# Consistent x-axis ticks across all subplots
fig_acf_pacf.update_xaxes(
    tickmode='linear', tick0=5, dtick=5, range=[0, MAX_LAGS + 1],
    title_text='Lag'
)

fig_acf_pacf.show()


# ── 5. Curve segment regime filter ────────────────────────────────────────────
def compute_segment_regime_filter(rolling_adf_df, mode='EOD'):
    """
    Compute a daily boolean flag per curve segment indicating whether
    that segment is in a non-stationary regime.

    Logic: if 2+ tenors in a segment have rolling ADF p-value > 0.05 on
    a given date, the whole segment is flagged True (blocked for new signals).
    This catches regime-wide dislocation versus idiosyncratic tenor moves.

    Segment definitions
    -------------------
    Short End   : 1Mo, 3Mo, 6Mo, 1Yr   (4 tenors — trigger at 2+)
    Belly Short : 2Yr, 3Yr, 5Yr        (3 tenors — trigger at 2+)
    Belly Long  : 7Yr, 10Yr            (2 tenors — trigger at 2+, i.e. both)
    Long End    : 30Yr                 (1 tenor  — 2+ threshold unreachable;
                                        30Yr is gated by its own rolling ADF)

    Parameters
    ----------
    rolling_adf_df : pd.DataFrame
        Rolling ADF p-values indexed by date. Columns must include all 10 tenors.
        Output of the 60-day rolling ADF computation in the stationarity cell.
    mode : str
        Data mode — 'EOD', 'intraday', or '5day'. Label only; no branching.

    Returns
    -------
    segment_regime_df : pd.DataFrame
        Boolean DataFrame indexed by date, four columns (one per segment).
        True  = non-stationary regime — no new signals for any tenor in segment.
        False = stationary regime     — signals eligible subject to other gates.
    """
    segments = {
        'Short End':   ['1Mo', '3Mo', '6Mo', '1Yr'],
        'Belly Short': ['2Yr', '3Yr', '5Yr'],
        'Belly Long':  ['7Yr', '10Yr'],
        'Long End':    ['30Yr'],           # single-tenor; flag never triggered via 2+ rule
    }

    regime = {}
    for seg_name, seg_tenors in segments.items():
        # Count non-stationary tenors per day (p > 0.05 means ADF fails to reject unit root)
        n_nonstationary = (rolling_adf_df[seg_tenors] > 0.05).sum(axis=1)
        regime[seg_name] = n_nonstationary >= 2

    return pd.DataFrame(regime, index=rolling_adf_df.index)


# ── Execute ────────────────────────────────────────────────────────────────────
segment_regime_df = compute_segment_regime_filter(rolling_adf_df, mode=mode)

_seg_tenors_map = {
    'Short End':   ['1Mo', '3Mo', '6Mo', '1Yr'],
    'Belly Short': ['2Yr', '3Yr', '5Yr'],
    'Belly Long':  ['7Yr', '10Yr'],
    'Long End':    ['30Yr'],
}

print("\nCurve Segment Regime Filter — Blocked-Day Summary")
print(f"Mode: {mode}  |  {segment_regime_df.index[0].date()} → {segment_regime_df.index[-1].date()}")
print(f"Rolling ADF window: {rolling_adf_window} days  |  Total dates: {len(segment_regime_df)}")
print()
print(f"{'Segment':<14}  {'Tenors':<28}  {'Blocked Days':>12}  {'% History':>10}")
print("─" * 72)
for seg_name, seg_tenors in _seg_tenors_map.items():
    blocked = int(segment_regime_df[seg_name].sum())
    pct     = blocked / len(segment_regime_df) * 100
    note    = ' *' if seg_name == 'Long End' else ''
    print(f"{seg_name:<14}  {', '.join(seg_tenors):<28}  {blocked:>12}  {pct:>9.1f}%{note}")
print("─" * 72)
print("* Long End (30Yr only): 2+ threshold unreachable — gated by per-tenor rolling ADF instead")
